# RAG Databricks Bluetab - LLM Model Registration

## Overview
This notebook registers a Large Language Model (LLM) for the RAG pipeline using Hugging Face transformers and MLflow.

## Features
- FLAN-T5 model registration with custom wrapper
- Chat API compatible output format
- Configurable model parameters and generation settings
- MLflow model packaging with dedicated requirements file
- Comprehensive testing and validation

## Model Details
- **Base Model**: google/flan-t5-base (configurable)
- **Task**: Text-to-text generation
- **Framework**: PyTorch + Transformers
- **API Format**: Compatible with LangChain ChatModel interface

## Dependencies
- Run `00 Configuration and Utils` notebook first
- Uses specific requirements file: `requirements_llm_model.txt`

## Requirements Management
This notebook uses a dedicated requirements file (`requirements_llm_model.txt`) for model registration, ensuring:
- ✅ **Reproducible deployments** with exact dependency versions
- ✅ **Isolated dependencies** specific to LLM models
- ✅ **Easier maintenance** and version control
- ✅ **Consistent serving environment** across deployments

In [0]:
# Instalar dependencias desde requirements.txt
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
# Core Configuration
dbutils.widgets.text("catalog_name", "bluetab", "Catalog Name")
dbutils.widgets.text("schema_name", "rag", "Schema Name")
dbutils.widgets.text("environment", "dev", "Environment (dev/test/prod)")

# Model Configuration
dbutils.widgets.text("llm_model_name", "flan_t5_base_model", "LLM Model Name")

# MLflow Configuration
dbutils.widgets.text("experiment_name", "/Shared/RAG_Databricks_Bluetab_Pipeline", "MLflow Experiment Name")

# MLflow run management
dbutils.widgets.text("parent_run_id", "", "Parent Run ID")
dbutils.widgets.text("current_run", "", "Current Run")

# LLM Configuration
dbutils.widgets.text("llm_base_model", "google/flan-t5-base", "Base LLM Model")
dbutils.widgets.text("max_length", "150", "Max Generation Length")
dbutils.widgets.text("temperature", "0.1", "Generation Temperature")
dbutils.widgets.dropdown("device", "cpu", ["cpu", "cuda", "auto"], "Device")
dbutils.widgets.text("model_suffix", "", "Model Name Suffix (optional)")

In [0]:

# Obtener valores de los widgets
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
ENVIRONMENT = dbutils.widgets.get("environment")

# Model names
LLM_MODEL_NAME = dbutils.widgets.get("llm_model_name")

# MLflow configuration
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

# Variables globales para gestión de parent/child runs
PARENT_RUN_ID = dbutils.widgets.get("parent_run_id") or None
CURRENT_RUN = dbutils.widgets.get("current_run") or None

LLM_MODEL_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{LLM_MODEL_NAME}"

LLM_BASE_MODEL = dbutils.widgets.get("llm_base_model")
MAX_LENGTH = int(dbutils.widgets.get("max_length"))
TEMPERATURE = float(dbutils.widgets.get("temperature"))
DEVICE = dbutils.widgets.get("device")
MODEL_SUFFIX = dbutils.widgets.get("model_suffix")

# Build model name with optional suffix
if MODEL_SUFFIX:
    LLM_MODEL_NAME_FULL = f"{LLM_MODEL_FULL}_{MODEL_SUFFIX}"
else:
    LLM_MODEL_NAME_FULL = LLM_MODEL_FULL

print("¡Configuración cargada correctamente!")
print(f"Environment: {ENVIRONMENT}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Schema: {SCHEMA_NAME}")


In [0]:
%run "./00 Configuration and Utils"

In [0]:
start_child_run("05_register_llm_model")

In [0]:
print(f"LLM Configuration:")
print(f"  Base Model: {LLM_BASE_MODEL}")
print(f"  Registered Name: {LLM_MODEL_NAME_FULL}")
print(f"  Max Length: {MAX_LENGTH}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Device: {DEVICE}")

In [0]:
import mlflow
import pandas as pd
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from mlflow.models.signature import infer_signature

# Start MLflow run for LLM registration
# with mlflow.start_run(run_name=f"05_Register_LLM_Model_{ENVIRONMENT}") as run:
mlflow.log_param("step", "llm_model_registration")
mlflow.log_param("environment", ENVIRONMENT)
mlflow.log_param("base_model", LLM_BASE_MODEL)
mlflow.log_param("max_length", MAX_LENGTH)
mlflow.log_param("temperature", TEMPERATURE)
mlflow.log_param("device", DEVICE)

log_step("llm_registration", "started", f"Registering {LLM_MODEL_NAME_FULL}")

# Load the base model and tokenizer
log_step("model_loading", "started", f"Loading {LLM_BASE_MODEL}")

try:
    tokenizer = AutoTokenizer.from_pretrained(LLM_BASE_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(LLM_BASE_MODEL)
    
    # Set device
    device_id = -1 if DEVICE == "cpu" else 0 if DEVICE == "cuda" else -1
    pipe = pipeline(
        "text2text-generation", 
        model=model, 
        tokenizer=tokenizer, 
        device=device_id
    )
    
    log_step("model_loading", "success", f"Model loaded on {DEVICE}")
    mlflow.log_param("model_loaded", True)
    
except Exception as e:
    log_step("model_loading", "failed", f"Error loading model: {e}")
    mlflow.log_param("model_loaded", False)
    raise e

In [0]:
# Define custom LLM wrapper for MLflow with Chat API compatibility
class FlanT5Wrapper(mlflow.pyfunc.PythonModel):
    """
    Custom MLflow wrapper for FLAN-T5 model with Chat API compatibility.
    
    This wrapper provides:
    - LangChain ChatModel compatible output format
    - Configurable generation parameters
    - Proper error handling and logging
    - Support for both single and batch inference
    """
    
    def load_context(self, context):
        """Load the model and tokenizer when the endpoint starts"""
        try:
            log_step("wrapper_load", "started", f"Loading {LLM_BASE_MODEL} in wrapper")
            
            from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
            
            # Load model components
            self.tokenizer = AutoTokenizer.from_pretrained(LLM_BASE_MODEL)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(LLM_BASE_MODEL)
            
            # Set device
            device_id = -1 if DEVICE == "cpu" else 0 if DEVICE == "cuda" else -1
            self.pipe = pipeline(
                "text2text-generation", 
                model=self.model, 
                tokenizer=self.tokenizer, 
                device=device_id
            )
            
            # Store generation parameters
            self.max_length = MAX_LENGTH
            self.temperature = TEMPERATURE
            
            log_step("wrapper_load", "success", "Model loaded in wrapper")
            
        except Exception as e:
            log_step("wrapper_load", "failed", f"Error loading model in wrapper: {e}")
            raise e

    def predict(self, context, model_input):
        """
        Generate responses using the LLM.
        
        Args:
            context: MLflow context (not used)
            model_input: DataFrame with 'messages' column containing chat history
            
        Returns:
            dict: Response in Chat API format with 'choices' key
        """
        try:
            # Extract prompts from chat messages
            # Each message should be a list of dicts with 'role' and 'content' keys
            prompts = []
            for chat_history in model_input["messages"].tolist():
                if isinstance(chat_history, list) and len(chat_history) > 0:
                    # Get the last user message
                    last_message = chat_history[-1]
                    if isinstance(last_message, dict) and 'content' in last_message:
                        prompts.append(last_message['content'])
                    else:
                        prompts.append(str(last_message))
                else:
                    prompts.append(str(chat_history))
            
            # Generate responses
            predictions_text = []
            for prompt in prompts:
                try:
                    result = self.pipe(
                        prompt, 
                        max_length=self.max_length, 
                        temperature=self.temperature,
                        do_sample=True if self.temperature > 0 else False
                    )
                    
                    if isinstance(result, list) and len(result) > 0:
                        generated_text = result[0].get("generated_text", "")
                    else:
                        generated_text = ""
                        
                    predictions_text.append(generated_text)
                    
                except Exception as e:
                    log_step("generation_error", "failed", f"Error generating response: {e}")
                    predictions_text.append(f"Error generating response: {str(e)}")
            
            # Build Chat API compatible response
            choices = []
            for text in predictions_text:
                choices.append({
                    "message": {
                        "role": "assistant",
                        "content": text
                    }
                })
            
            return {"choices": choices}
            
        except Exception as e:
            log_step("predict_error", "failed", f"Error in prediction: {e}")
            # Return error response in Chat API format
            return {
                "choices": [{
                    "message": {
                        "role": "assistant", 
                        "content": f"Error: {str(e)}"
                    }
                }]
            }

print("✅ LLM Wrapper class defined successfully")

In [0]:
# Create test input and output for signature inference
log_step("signature_creation", "started", "Creating model signature")

try:
    # Create test input in Chat API format
    contexto = "La paella valenciana es un plato de arroz originario de la Comunidad Valenciana."
    pregunta = "¿De dónde es la paella?"
    prompt_completo = f"Contexto: {contexto}\n\nPregunta: {pregunta}\n\nRespuesta:"
    
    input_data = [[{
        'role': 'user', 
        'content': prompt_completo
    }]]
    
    input_example = pd.DataFrame({"messages": input_data})
    
    # Generate test output using the pipeline
    test_prompt = input_data[0][0]['content']
    test_result = pipe(test_prompt, max_length=MAX_LENGTH, temperature=TEMPERATURE)[0]["generated_text"]
    
    # Create expected output format
    output_example = {
        "choices": [
            {
                "message": {
                    "role": "assistant",
                    "content": test_result
                }
            }
        ]
    }
    
    # Infer signature
    signature = infer_signature(input_example, output_example)
    
    log_step("signature_creation", "success", "Model signature created")
    
    print("📝 Test Input/Output:")
    print(f"   Input: {prompt_completo[:100]}...")
    print(f"   Output: {test_result[:100]}...")
    print("\n✅ Model signature inferred successfully")
    
    # Log test details
    mlflow.log_param("test_input_format", "chat_api")
    mlflow.log_param("test_output_format", "chat_api")
    mlflow.log_text(test_prompt, "test_prompt.txt")
    mlflow.log_text(test_result, "test_response.txt")
    
except Exception as e:
    log_step("signature_creation", "failed", f"Error creating signature: {e}")
    raise e

In [0]:
# Register the model with MLflow
log_step("model_registration", "started", f"Registering model {LLM_MODEL_NAME_FULL}")

try:
    # Use the specific requirements file for LLM model
    requirements_file_path = "../requirements_llm_model.txt"

    mlflow.set_registry_uri("databricks-uc")
    
    model_info = mlflow.pyfunc.log_model(
        artifact_path="flan_t5_model",
        python_model=FlanT5Wrapper(),
        registered_model_name=LLM_MODEL_NAME_FULL,
        input_example=input_example,
        signature=signature,
        pip_requirements=requirements_file_path,
        metadata={
            "base_model": LLM_BASE_MODEL,
            "max_length": MAX_LENGTH,
            "temperature": TEMPERATURE,
            "device": DEVICE,
            "task": "text2text-generation",
            "api_format": "chat",
            "framework": "transformers",
            "environment": ENVIRONMENT,
            "requirements_file": requirements_file_path
        }
    )
    
    # Log registration results
    mlflow.log_param("registration_status", "success")
    mlflow.log_param("model_uri", model_info.model_uri)
    mlflow.log_param("model_version", model_info.registered_model_version)
    mlflow.log_param("requirements_source", "requirements_llm_model.txt")
    
    log_step("model_registration", "success", f"Model registered as version {model_info.registered_model_version}")
    
    print(f"✅ Model '{LLM_MODEL_NAME_FULL}' registered successfully!")
    print(f"📍 Model URI: {model_info.model_uri}")
    print(f"🔢 Version: {model_info.registered_model_version}")
    print(f"📄 Requirements: {requirements_file_path}")
    
except Exception as e:
    mlflow.log_param("registration_status", "failed")
    mlflow.log_param("registration_error", str(e))
    
    log_step("model_registration", "failed", f"Error registering model: {e}")
    print(f"❌ Error registering model: {e}")
    raise e

In [0]:
# Test the registered model
log_step("model_testing", "started", "Testing registered model")

try:
    # Load the registered model
    model_version = model_info.registered_model_version if 'model_info' in locals() else "latest"
    model_uri = f"models:/{LLM_MODEL_NAME_FULL}/{model_version}"
    
    loaded_model = mlflow.pyfunc.load_model(model_uri)
    
    log_step("model_load_test", "success", f"Loaded model version {model_version}")
    
    # Create test data for RAG scenario
    test_contexts = [
        "La paella valenciana es un plato de arroz originario de la Comunidad Valenciana. Sus ingredientes tradicionales incluyen arroz, pollo, conejo y verduras locales.",
        "El gazpacho es una sopa fría originaria de Andalucía. Sus ingredientes principales son tomate, pepino, pimiento, cebolla, ajo, aceite de oliva, vinagre y pan."
    ]
    
    test_questions = [
        "¿Qué ingredientes lleva la paella?",
        "¿Qué ingredientes lleva el gazpacho?"
    ]
    
    test_messages = []
    for context, question in zip(test_contexts, test_questions):
        prompt = f"Contexto: {context}\n\nPregunta: {question}\n\nResponde basándote únicamente en el contexto.\nRespuesta:"
        test_messages.append([{"role": "user", "content": prompt}])
    
    test_df = pd.DataFrame({"messages": test_messages})
    
    # Make predictions
    predictions = loaded_model.predict(test_df)
    
    # Validate response format
    assert "choices" in predictions, "Response must contain 'choices' key"
    assert len(predictions["choices"]) == len(test_messages), "Should have one choice per input"
    
    # Log test results
    mlflow.log_metric("test_inputs_count", len(test_messages))
    mlflow.log_metric("test_predictions_count", len(predictions["choices"]))
    mlflow.log_param("test_status", "passed")
    
    log_step("model_testing", "success", f"Model test passed - {len(predictions['choices'])} responses generated")
    
    print("🧪 Model Test Results:")
    print(f"   Test cases: {len(test_messages)}")
    print(f"   Responses generated: {len(predictions['choices'])}")
    
    # Display sample responses
    for i, choice in enumerate(predictions["choices"][:2]):  # Show first 2
        response = choice["message"]["content"]
        print(f"\n   Test {i+1} Response: {response[:100]}{'...' if len(response) > 100 else ''}")
    
    print("   ✅ All tests passed!")
    
except Exception as e:
    mlflow.log_param("test_status", "failed")
    mlflow.log_param("test_error", str(e))
    
    log_step("model_testing", "failed", f"Model test failed: {e}")
    print(f"❌ Model test failed: {e}")

In [0]:
# Final summary
log_step("llm_registration", "completed", "LLM registration process finished")

print("="*60)
print("LLM MODEL REGISTRATION SUMMARY")
print("="*60)
print(f"Model Name: {LLM_MODEL_NAME_FULL}")
print(f"Base Model: {LLM_BASE_MODEL}")
print(f"Environment: {ENVIRONMENT}")
print(f"Max Length: {MAX_LENGTH}")
print(f"Temperature: {TEMPERATURE}")
print(f"Device: {DEVICE}")

if 'model_info' in locals():
    print(f"Version: {model_info.registered_model_version}")
    print(f"URI: {model_info.model_uri}")
    print("✅ Registration: SUCCESS")
else:
    print("❌ Registration: FAILED")

print("="*60)